# 09 — Reconciliation & Validation

**Purpose:** Validate Bronze → Silver → Gold after running the normal Gold Databricks notebooks (07 and 08).

This notebook is for checking only. It does not create, overwrite, or modify any tables.

**Run order:** Run notebooks 01–06, then run notebook 07 (Gold dimensions), notebook 08 (Gold facts and KPIs), and finally this notebook.

Checks included:
- Bronze = Silver + Quarantine for each source, where a quarantine table exists.
- Gold dimensions match Silver master counts and have unique business keys.
- Gold facts match their Silver event counts and retain their event-grain keys.
- Transaction fact references valid dimensions without row multiplication.
- Transaction amount totals reconcile between Silver and Gold.

A failed check is reported for investigation; it does not automatically mean the source data is wrong.


In [0]:
from pyspark.sql.functions import col, count, sum as sum_

BRONZE = "capstone_team01.bronze_schema"
SILVER = "capstone_team01.silver_schema"
GOLD = "capstone_team01.gold_schema"


## 1. Helper for checking table availability

Gold Delta tables created by notebooks 07 and 08 must already exist. If a table is missing, run/fix the relevant Gold notebook before continuing.

In [0]:
def table_exists(table_name):
    return spark.catalog.tableExists(table_name)

def get_count(table_name):
    if table_exists(table_name):
        return spark.table(table_name).count()
    return None

def quarantine_count(table_name):
    if table_exists(table_name):
        return spark.table(table_name).count()
    return 0


## 2. Bronze to Silver reconciliation

A quarantine table is optional. If no records were rejected, its count is treated as zero.

In [0]:
reconciliation = [
    ("card_transactions", "bronze_card_transactions", "silver_card_transactions", "quarantine_card_transactions"),
    ("customer_master", "bronze_customer_master", "silver_customer_master", "quarantine_customer_master"),
    ("merchant_master", "bronze_merchant_master", "silver_merchant_master", "quarantine_merchant_master"),
    ("card_master", "bronze_card_master", "silver_card_master", "quarantine_card_master"),
    ("device_authentication_events", "bronze_device_authentication_events", "silver_device_authentication_events", "quarantine_device_authentication_events"),
    ("payment_status_changes", "bronze_payment_status_changes", "silver_payment_status_changes", "quarantine_payment_status_changes")
]

reconciliation_results = []

for name, bronze_name, silver_name, quarantine_name in reconciliation:
    bronze_table = f"{BRONZE}.{bronze_name}"
    silver_table = f"{SILVER}.{silver_name}"
    quarantine_table = f"{SILVER}.{quarantine_name}"

    bronze_rows = get_count(bronze_table)
    silver_rows = get_count(silver_table)
    quarantine_rows = quarantine_count(quarantine_table)

    reconciled = (
        bronze_rows is not None and
        silver_rows is not None and
        bronze_rows == silver_rows + quarantine_rows
    )

    reconciliation_results.append((
        name, bronze_rows, silver_rows, quarantine_rows, reconciled
    ))

reconciliation_df = spark.createDataFrame(
    reconciliation_results,
    ["source", "bronze_count", "silver_count", "quarantine_count", "reconciled"]
)

display(reconciliation_df)


## 3. Gold table availability

Confirm that notebooks 07 and 08 created all expected Gold dimensions, facts and KPI tables.

In [0]:
gold_tables = [
    "dim_customer",
    "dim_merchant",
    "dim_card",
    "fact_card_transactions",
    "fact_authentication_events",
    "fact_payment_status_changes",
    "gold_merchant_payment_volume",
    "gold_authentication_patterns"
]

gold_availability = []
for name in gold_tables:
    full_name = f"{GOLD}.{name}"
    gold_availability.append((name, table_exists(full_name), get_count(full_name)))

availability_df = spark.createDataFrame(
    gold_availability,
    ["gold_object", "exists", "row_count"]
)
display(availability_df)


## 4. Dimension validation

Each dimension should have one row per business key and should match its Silver master source count.

In [0]:
dimension_checks = [
    ("dim_customer", "silver_customer_master", "customer_id"),
    ("dim_merchant", "silver_merchant_master", "merchant_id"),
    ("dim_card", "silver_card_master", "card_id")
]

dimension_results = []

for gold_name, silver_name, key in dimension_checks:
    gold_table = f"{GOLD}.{gold_name}"
    silver_table = f"{SILVER}.{silver_name}"

    if table_exists(gold_table) and table_exists(silver_table):
        gold_df = spark.table(gold_table)
        silver_rows = spark.table(silver_table).count()
        gold_rows = gold_df.count()
        duplicate_keys = (gold_df.groupBy(key).count()
                          .filter(col("count") > 1).count())
        null_keys = gold_df.filter(col(key).isNull()).count()
        count_match = gold_rows == silver_rows
    else:
        silver_rows = None
        gold_rows = None
        duplicate_keys = None
        null_keys = None
        count_match = False

    dimension_results.append((
        gold_name, silver_rows, gold_rows, duplicate_keys, null_keys, count_match
    ))

dimension_df = spark.createDataFrame(
    dimension_results,
    ["dimension", "silver_count", "gold_count", "duplicate_keys", "null_keys", "count_matches"]
)
display(dimension_df)


## 5. Fact count and event-grain validation

Expected grains:
- Transaction fact: one row per `transaction_id`.
- Authentication fact: one row per `auth_event_id`.
- Payment status fact: one row per `change_event_id`.

Do not use `transaction_id` as the payment-status event key; one transaction can have multiple status-change events.

In [0]:
fact_checks = [
    ("fact_card_transactions", "silver_card_transactions", "transaction_id"),
    ("fact_authentication_events", "silver_device_authentication_events", "auth_event_id"),
    ("fact_payment_status_changes", "silver_payment_status_changes", "change_event_id")
]

fact_results = []

for gold_name, silver_name, key in fact_checks:
    gold_table = f"{GOLD}.{gold_name}"
    silver_table = f"{SILVER}.{silver_name}"

    if table_exists(gold_table) and table_exists(silver_table):
        fact_df = spark.table(gold_table)
        silver_rows = spark.table(silver_table).count()
        gold_rows = fact_df.count()
        duplicate_keys = (fact_df.groupBy(key).count()
                          .filter(col("count") > 1).count())
        null_keys = fact_df.filter(col(key).isNull()).count()
        count_match = gold_rows == silver_rows
    else:
        silver_rows = None
        gold_rows = None
        duplicate_keys = None
        null_keys = None
        count_match = False

    fact_results.append((
        gold_name, silver_rows, gold_rows, duplicate_keys, null_keys, count_match
    ))

fact_df = spark.createDataFrame(
    fact_results,
    ["fact", "silver_count", "gold_count", "duplicate_event_keys", "null_event_keys", "count_matches"]
)
display(fact_df)


## 6. Transaction fact reference validation

Check that the transaction fact contains valid customer, merchant and card references. These checks also help identify accidental row multiplication caused by non-unique dimension keys.

In [0]:
if table_exists(f"{GOLD}.fact_card_transactions"):
    transaction_fact = spark.table(f"{GOLD}.fact_card_transactions")

    customer_keys = spark.table(f"{GOLD}.dim_customer").select("customer_id").dropDuplicates()
    merchant_keys = spark.table(f"{GOLD}.dim_merchant").select("merchant_id").dropDuplicates()
    card_keys = spark.table(f"{GOLD}.dim_card").select("card_id").dropDuplicates()

    missing_customers = transaction_fact.join(customer_keys, "customer_id", "left_anti").count()
    missing_merchants = transaction_fact.join(merchant_keys, "merchant_id", "left_anti").count()
    missing_cards = transaction_fact.join(card_keys, "card_id", "left_anti").count()

    print("Transactions with missing customer dimension:", missing_customers)
    print("Transactions with missing merchant dimension:", missing_merchants)
    print("Transactions with missing card dimension:", missing_cards)
else:
    print("fact_card_transactions is not available. Run notebook 08 first.")


## 7. Transaction amount reconciliation

Compare transaction count and total amount between Silver and the Gold transaction fact. This helps detect dropped or multiplied rows during Gold transformations.

In [0]:
if (table_exists(f"{SILVER}.silver_card_transactions") and
    table_exists(f"{GOLD}.fact_card_transactions")):

    silver_transactions = spark.table(f"{SILVER}.silver_card_transactions")
    gold_transactions = spark.table(f"{GOLD}.fact_card_transactions")

    silver_summary = silver_transactions.agg(
        count("transaction_id").alias("row_count"),
        sum_("amount").alias("total_amount")
    ).first()

    gold_summary = gold_transactions.agg(
        count("transaction_id").alias("row_count"),
        sum_("amount").alias("total_amount")
    ).first()

    print("Silver transaction rows:", silver_summary["row_count"])
    print("Gold transaction rows:", gold_summary["row_count"])
    print("Silver transaction amount:", silver_summary["total_amount"])
    print("Gold transaction amount:", gold_summary["total_amount"])
    print("Row count matches:", silver_summary["row_count"] == gold_summary["row_count"])
    print("Amount matches:", silver_summary["total_amount"] == gold_summary["total_amount"])
else:
    print("Silver or Gold transaction table is missing.")


## 8. KPI aggregate sanity checks

These are structural checks only. The KPI definitions and classifications must be validated separately against approved payment-status and risk-signal meanings. This notebook does not guess which values mean authorized, failed or suspicious.

In [0]:
if table_exists(f"{GOLD}.gold_merchant_payment_volume"):
    print("Merchant/payment-status aggregate:")
    display(
        spark.table(f"{GOLD}.gold_merchant_payment_volume")
        .orderBy(col("transaction_count").desc())
        .limit(20)
    )

if table_exists(f"{GOLD}.gold_authentication_patterns"):
    print("Authentication pattern aggregate:")
    display(
        spark.table(f"{GOLD}.gold_authentication_patterns")
        .orderBy(col("authentication_event_count").desc())
    )


## 9. Final validation summary

Review all displayed results. Expected conditions:
- Bronze reconciliation is true for each source.
- Dimension and fact row counts match their Silver sources.
- Dimension and fact business/event keys have zero duplicates and zero nulls.
- Transaction fact has zero missing dimension references.
- Silver and Gold transaction counts and amount totals match.

Record any failures and investigate before treating Gold as ready for dashboard or ML use. A successful count check alone does not certify KPI business semantics.

In [0]:
print("Validation notebook completed.")
print("Review the reconciliation, dimension, fact, reference and amount checks above.")
